# Packages and Misc

In [5]:
import pandas as pd
import requests
import json
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
mpl.rcParams['figure.figsize'] = (10,7) 
import seaborn as sns
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.interpolate import interp1d
from tqdm.notebook import tqdm
from statsmodels.tsa.seasonal import STL
import requests
import json
import numpy as np
import re
import plotly.express as ex
from airrship.create_repertoire import generate_sequence,load_data,get_genotype,create_allele_dict
import importlib
import plotly.io as pio
tqdm.pandas()
# from Bio.Align.Applications import ClustalOmegaCommandline
# from Bio import AlignIO, SeqIO
# from Bio.SeqRecord import SeqRecord
# from Bio.Seq import Seq
import os
from scipy.stats import entropy
import matplotlib as mpl
from collections import defaultdict
import numpy as np

mpl.rcParams['figure.figsize'] = (20,11)
sns.set_context('poster')
tokenizer_dictionary = {
            "A": 1,
            "T": 2,
            "G": 3,
            "C": 4,
            "N": 5,
            "P": 0,  # pad token
        }
import pickle

def _process_and_dpad(sequence, train=True):
    start, end = None, None
    trans_seq = [tokenizer_dictionary[i] for i in sequence]

    gap = max_length - len(trans_seq)
    iseven = gap % 2 == 0
    whole_half_gap = gap // 2

    if iseven:
        trans_seq = [0] * whole_half_gap + trans_seq + ([0] * whole_half_gap)
        if train:
            start, end = whole_half_gap, max_length - whole_half_gap - 1

    else:
        trans_seq = [0] * (whole_half_gap + 1) + trans_seq + ([0] * whole_half_gap)
        if train:
            start, end = (whole_half_gap + 1, max_length - whole_half_gap - 1)

    return trans_seq, start, end if iseven else (end + 1)

def process_sequences(self, data: pd.DataFrame, corrupt_beginning=False, verbose=False):
    padded_sequences = []
    v_start, v_end, d_start, d_end, j_start, j_end = [], [], [], [], [], []
    iterator = tqdm(data.itertuples(), total=len(data)) if verbose else data.itertuples()
    for row in iterator:
        seq = row.sequence
        padded_array, start, end = _process_and_dpad(seq, self.max_length)
        padded_sequences.append(padded_array)
        _adjust = start


        v_start.append(start)
        j_end.append(end)
        v_end.append(row.v_sequence_end + _adjust)
        d_start.append(row.d_sequence_start + _adjust)
        d_end.append(row.d_sequence_end + _adjust)
        j_start.append(row.j_sequence_start + _adjust)

    v_start = np.array(v_start)
    v_end = np.array(v_end)
    d_start = np.array(d_start)
    d_end = np.array(d_end)
    j_start = np.array(j_start)
    j_end = np.array(j_end)

    padded_sequences = np.vstack(padded_sequences)

    return v_start, v_end, d_start, d_end, j_start, j_end, padded_sequences
def global_genotype():
    try:
        path_to_data = importlib.resources.files(
            'airrship').joinpath("data")
    except AttributeError:
        with importlib.resources.path('airrship', 'data') as p:
            path_to_data = p
    v_alleles = create_allele_dict(
        f"{path_to_data}/imgt_human_IGHV.fasta")
    d_alleles = create_allele_dict(
        f"{path_to_data}/imgt_human_IGHD.fasta")
    j_alleles = create_allele_dict(
        f"{path_to_data}/imgt_human_IGHJ.fasta")

    vdj_allele_dicts = {"V": v_alleles,
                        "D": d_alleles,
                        "J": j_alleles}

    chromosome1, chromosome2 = defaultdict(list), defaultdict(list)
    for segment in ["V", "D", "J"]:
        allele_dict = vdj_allele_dicts[segment]
        for gene in allele_dict.values():
            for allele in gene:
                chromosome1[segment].append(allele)
                chromosome2[segment].append(allele)

    locus = [chromosome1, chromosome2]
    return locus


def decompose_call(call):
    family, G = call.split("-", 1)
    gene, allele = G.split("*")
    return family, gene, allele
locus = global_genotype()

from VDeepJUnbondedDataset import global_genotype

locus = global_genotype()
v_dict = {i.name: i.ungapped_seq.upper() for i in locus[0]['V']}
d_dict = {i.name: i.ungapped_seq.upper() for i in locus[0]['D']}
j_dict = {i.name: i.ungapped_seq.upper() for i in locus[0]['J']}
        
v_alleles = sorted(list(v_dict))
d_alleles = sorted(list(d_dict))
j_alleles = sorted(list(j_dict))

v_allele_count = len(v_alleles)
d_allele_count = len(d_alleles)
j_allele_count = len(j_alleles)


v_allele_call_ohe = {f: i for i, f in enumerate(v_alleles)}
d_allele_call_ohe = {f: i for i, f in enumerate(d_alleles)}
j_allele_call_ohe = {f: i for i, f in enumerate(j_alleles)}

v_allele_call_rev_ohe = {i: f for i, f in enumerate(v_alleles)}
d_allele_call_rev_ohe = {i: f for i, f in enumerate(d_alleles)}
j_allele_call_rev_ohe = {i: f for i, f in enumerate(j_alleles)}

def encode_igb_v_call(v_call):
    v = np.zeros(len(v_allele_call_rev_ohe))
    for i in v_call.split(','):
        v[v_allele_call_ohe[i]] = 1
    return v

In [13]:
# for data generation
from airrship.create_repertoire import generate_sequence,load_data,get_genotype,create_allele_dict

data_dict = load_data()

def predict_sample(sample):
    eval_dataset_ = trainer.train_dataset.tokenize_sequences([sample])
    padded_seqs_tensor = tf.convert_to_tensor(eval_dataset_, dtype=tf.int32)
    dataset_from_tensors = tf.data.Dataset.from_tensor_slices({
        'tokenized_sequence': padded_seqs_tensor,
        'tokenized_sequence_for_masking': padded_seqs_tensor
    })
    dataset = (
        dataset_from_tensors
        .batch(1)
        .prefetch(tf.data.AUTOTUNE)
    )

    predicted = trainer.model.predict(dataset, verbose=True)
    return predicted
def get_v_latent_projection(sequence):
    eval_dataset_ = trainer.train_dataset.tokenize_sequences([sequence])
    v_mask_input_embedding = trainer.model.concatenated_v_mask_input_embedding(
        eval_dataset_
    )
    v_feature_map = trainer.model._encode_masked_v_signal(v_mask_input_embedding)

    v_allele_latent = trainer.model.v_allele_mid(v_feature_map)
    v_allele_latent = dec.transform(v_allele_latent.numpy())
    return v_allele_latent[0]

def getting_padding_size(seq, max_length=512):
    start, end = None, None
    gap = max_length - len(seq)
    iseven = gap % 2 == 0
    whole_half_gap = gap // 2

    if iseven:
        start, end = whole_half_gap, whole_half_gap

    else:
        start, end = whole_half_gap + 1, whole_half_gap
    return start, end
def log_threshold(prediction,th=0.4):
    ast = np.argsort(prediction)[::-1]
    R = [ast[0]]
    for ip in range(1,len(ast)):
        DIFF = np.log(prediction[ast[ip-1]]/prediction[ast[ip]])
        if DIFF<th:
            R.append(ast[ip])
        else:
            break
    return R
def extract_prediction_alleles(probabilites,th=0.4):
    V_ratio = []
    for v_all in (probabilites):
        v_alleles  = log_threshold(v_all,th=th)
        V_ratio.append([v_allele_call_rev_ohe[i] for i in v_alleles])
    return V_ratio
def allign_sequences(seqs):
    sequences = [
    SeqRecord(Seq(seq), id=f"seq{en}")
    for en,seq in enumerate(seqs)
    ]
    SeqIO.write(sequences, r"C:\Users\Tomas\Downloads\clustal-omega-1.2.2-win64\clustal-omega-1.2.2-win64\sequences.fasta", "fasta")

    # Define Clustal Omega command
    clustalomega_cline = ClustalOmegaCommandline(infile=r"C:\Users\Tomas\Downloads\clustal-omega-1.2.2-win64\clustal-omega-1.2.2-win64\sequences.fasta",
                                                 outfile=r"C:\Users\Tomas\Downloads\clustal-omega-1.2.2-win64\clustal-omega-1.2.2-win64\aligned.fasta", verbose=True, auto=True,force=True)
    clustalomega_cline.program_name = r"C:\Users\Tomas\Downloads\clustal-omega-1.2.2-win64\clustal-omega-1.2.2-win64\clustalo.exe"
    stdout, stderr = clustalomega_cline()

    alignment = AlignIO.read(r"C:\Users\Tomas\Downloads\clustal-omega-1.2.2-win64\clustal-omega-1.2.2-win64\aligned.fasta", "fasta")
    return alignment

# Plot Misc

In [14]:
from IPython.display import display,HTML
from matplotlib.colors import LinearSegmentedColormap

def v_segment_igb_plot(sequence,igb_v_start,igb_v_end,v_segment):
    colormap = plt.get_cmap("coolwarm")

    def get_color(v_segment_value):
        return colormap(v_segment_value)

    colored_sequence = [
        f'<span style="color: {mpl.colors.to_hex(get_color(v))};font-size:20px;font-family: "Times New Roman", Times, serif;">{char}</span>'
        for char, v in zip(sequence, v_segment)
    ]
    
    colored_sequence[igb_v_start] = colored_sequence[igb_v_start].replace('<span style="','<span style=" background-color:rgba(75, 255, 51, 0.44);font-weight:bold;').replace('font-size:20px;','font-size:25px;')
    colored_sequence[igb_v_end] = colored_sequence[igb_v_end].replace('<span style="','<span style=" background-color:rgba(255, 255, 255, 0.69);font-weight:bold;').replace('font-size:20px;','font-size:25px;')

    html = f"""
    <html style="background-color: white;">
    <head>
    <style>
    .colored-sequence-div {{
        width: 100%;            /* Full width of the container */
        word-wrap: break-word;  /* Allow long words to break and wrap */
    }}
    </style>
    </head>
    <body>
    <div class="colored-sequence-div">
    {''.join(colored_sequence)}
    </div>
    </body>
    </head>
    </html>
    """

    display(HTML(html))
    
    
idx = 15
v_segment_igb_plot(igb_predicted.sequence.iloc[idx],
                  igb_predicted.v_sequence_start.iloc[idx],
                  igb_predicted.v_sequence_end.iloc[idx],
                  v_segment_unpadded[idx])    

NameError: name 'igb_predicted' is not defined

In [15]:
import matplotlib.pyplot as plt
import matplotlib as mpl
from IPython.display import display, HTML

def multi_segment_plot(sequence, v_segment, d_segment, j_segment):
    # Define colormaps for each segment
    v_colormap = plt.get_cmap("Reds")
    d_colormap = plt.get_cmap("Greens")
    j_colormap = plt.get_cmap("Blues")

    # Helper function to get color based on segment value and colormap
    def get_color(segment_value, colormap):
        return colormap(segment_value)

    # Create a colored sequence; this assumes v_segment, d_segment, and j_segment are normalized to [0, 1]
    colored_sequence = []
    for char, v, d, j in zip(sequence, v_segment, d_segment, j_segment):
        # Determine which segment has the maximum value for this character
        max_segment = max(v, d, j)
        if max_segment == v:
            color = mpl.colors.to_hex(get_color(v, v_colormap))
        elif max_segment == d:
            color = mpl.colors.to_hex(get_color(d, d_colormap))
        elif max_segment==j:  # max_segment == j
            color = mpl.colors.to_hex(get_color(j, j_colormap))
            
        if max_segment < 0.01:
            styled_char = f'<span style="color: {"rgba(147, 143, 145, 0.31)"}; font-size:20px;font-family: Arial, Helvetica, sans-serif;"><strike>{char}</strike></span>'
        else:
            styled_char = f'<span style="color: {color}; font-size:20px;font-family: Arial, Helvetica, sans-serif;">{char}</span>'
        
        colored_sequence.append(styled_char)

    # Construct the HTML for display
    html_content = f"""
    <html style="background-color: white;">
    <head>
    <style>
    .colored-sequence-div {{
        width: 100%;            /* Full width of the container */
        word-wrap: break-word;  /* Allow long words to be hyphenated and wrapped */
        background-color: white;
        color:black;
    }}
    </style>
    </head>
    <body >
    <div class="colored-sequence-div">
    {''.join(colored_sequence)}
    </div>
    </body>
    </html>
    """

    # Display the HTML content
    display(HTML(html_content))
    
idx = 99
multi_segment_plot(igb_predicted.sequence.iloc[idx],
                  v_segment_unpadded[idx],
                  d_segment_unpadded[idx],
                  j_segment_unpadded[idx])    

NameError: name 'igb_predicted' is not defined

# Generate A Single Sequenece Using AIRRShip

In [16]:
import random
vo_dict # <- this has a list of all V alleles
for allele in (vo_dict):
    random_d = random.choice(locus[0]['D']) # a random D allele
    random_j = random.choice(locus[0]['J']) # a random J allele
    generated_seq = generate_sequence_spesific(vo_dict[allele],random_d,random_j,data_dict, mutate=True,
                                mutation_rate=0.01,# percentage of mutations
                                               shm_flat=False,
                                               flat_usage='allele')
    


NameError: name 'vo_dict' is not defined

# Predict

In [17]:
import tensorflow as tf
tokenizer_dictionary = {
            "A": 1,
            "T": 2,
            "G": 3,
            "C": 4,
            "N": 5,
            "P": 0,  # pad token
        }

max_seq_length = 512
def _process_and_dpad(sequence, train=True):
    """
    Private method, converts sequences into 4 one hot vectors and paddas them from both sides with zeros
    equal to the diffrenece between the max length and the sequence length
    :param nuc:
    :param self.max_seq_length:
    :return:
    """

    start, end = None, None
    trans_seq = [tokenizer_dictionary[i] for i in sequence]

    gap = max_seq_length - len(trans_seq)
    iseven = gap % 2 == 0
    whole_half_gap = gap // 2

    if iseven:
        trans_seq = [0] * whole_half_gap + trans_seq + ([0] * whole_half_gap)
        if train:
            start, end = whole_half_gap, max_seq_length - whole_half_gap - 1

    else:
        trans_seq = [0] * (whole_half_gap + 1) + trans_seq + ([0] * whole_half_gap)
        if train:
            start, end = (
                whole_half_gap + 1,
                max_seq_length - whole_half_gap - 1,
            )

    return trans_seq, start, end if iseven else (end + 1)
def preprocess_data(data):
    # Implement your data preprocessing here
    # This function should take an individual data sample or a batch of data
    # and return the preprocessed data
    return np.vstack([_process_and_dpad(i) for i in data])
def create_dataset(data, batch_size=128):
    dataset = tf.data.Dataset.from_tensor_slices(data)
    dataset = dataset.batch(batch_size)
    dataset = dataset.map(preprocess_data, num_parallel_calls=tf.data.AUTOTUNE)
    return dataset
def predict_in_batches(model, dataset):
    raw_predictions = []
    for batch_data in dataset:
        batch_preds = model.predict(batch_data, verbose=True)
        raw_predictions.extend(batch_preds)
    return raw_predictions



In [2]:
import tensorflow as tf
from VDeepJModelExperimental import VDeepJAllignExperimentalSingleBeamConvSegmentationResidualRF
from Trainer import SingleBeamSegmentationTrainer

# Start a trainer with the model
trainer = SingleBeamSegmentationTrainer(
    model=VDeepJAllignExperimentalSingleBeamConvSegmentationResidualRF,
    data_path = r"/localdata/alignairr_data/1M_for_test/sim_data_1M_asc_new_param_s5f_opposite_rate_02_add_n.tsv", # give it any small CSV dataset just for it to initialize
    batch_read_file=True,
    epochs=1,
    batch_size=1,
    steps_per_epoch=150_000,
    verbose=1,
    corrupt_beginning=True,
    classification_head_metric=[tf.keras.metrics.AUC(),tf.keras.metrics.AUC(),tf.keras.metrics.AUC()],
    interval_head_metric=tf.keras.losses.mae,
    corrupt_proba=0.7,
    airrship_mutation_rate=0.25,
    nucleotide_add_coef=210,
    nucleotide_remove_coef=330,
    random_sequence_add_proba=0.45,
    single_base_stream_proba=0.05,
    duplicate_leading_proba=0.25,
    random_allele_proba=0.25,
    num_parallel_calls=32,
)


In [1]:
#trainer.model.plot_model((512,1)) if you wamt to see the model builed 

In [3]:
# load the weights

trainer.model.build({'tokenized_sequence':(512,1)})
trainer.model.load_weights("/localdata/alignairr_data/sf5_alignairr_segmentation_residual_s_v_d_j_embedding_product/saved_models/sf5_alignairr_segmentation_residual_s_v_d_j_embedding_product_cp1")
print('Model Loaded!')



Model Loaded!


In [6]:
# Read the data from the TSV file
data = pd.read_table(r"/localdata/alignairr_data/naive_repertoires/naive_sequences_clean.tsv",usecols=['sequence'])



In [7]:
eval_dataset_ = trainer.train_dataset.tokenize_sequences(data.sequence.to_list())
print('Train Dataset Encoded!')

padded_seqs_tensor = tf.convert_to_tensor(eval_dataset_, dtype=tf.uint8)
dataset_from_tensors = tf.data.Dataset.from_tensor_slices({
    'tokenized_sequence': padded_seqs_tensor})


Train Dataset Encoded!


In [10]:
eval_dataset_[0]

array([0, 0, 0, 0, 0, 0, 3, 1, 3, 2, 3, 4, 2, 2, 2, 4, 2, 3, 1, 3, 1, 3,
       2, 4, 1, 2, 3, 3, 1, 4, 4, 2, 4, 4, 2, 3, 4, 1, 4, 1, 1, 3, 1, 1,
       4, 1, 2, 3, 1, 1, 1, 4, 1, 4, 4, 2, 3, 2, 3, 3, 2, 2, 4, 2, 2, 4,
       4, 2, 4, 4, 2, 4, 4, 2, 3, 3, 2, 3, 3, 4, 1, 3, 4, 2, 4, 4, 4, 1,
       3, 1, 2, 3, 3, 3, 2, 4, 4, 2, 3, 2, 4, 4, 4, 1, 3, 3, 2, 3, 4, 1,
       3, 4, 2, 1, 4, 1, 3, 4, 1, 3, 2, 3, 3, 3, 3, 4, 3, 4, 1, 3, 3, 1,
       4, 2, 3, 2, 2, 3, 1, 1, 3, 4, 4, 2, 2, 4, 3, 3, 1, 3, 1, 4, 4, 4,
       2, 3, 2, 4, 4, 4, 2, 4, 1, 4, 4, 2, 3, 4, 3, 4, 2, 3, 2, 4, 2, 1,
       2, 3, 3, 2, 3, 3, 3, 2, 4, 4, 2, 2, 4, 1, 3, 2, 3, 3, 2, 2, 5, 4,
       2, 4, 4, 2, 3, 3, 1, 3, 4, 2, 3, 3, 1, 2, 4, 4, 3, 4, 4, 1, 3, 5,
       4, 4, 4, 4, 1, 3, 3, 3, 1, 1, 3, 3, 3, 3, 4, 2, 3, 3, 1, 3, 2, 3,
       3, 1, 2, 2, 3, 3, 3, 3, 1, 1, 1, 2, 4, 1, 1, 2, 4, 1, 2, 1, 3, 2,
       3, 3, 1, 1, 3, 4, 1, 4, 4, 1, 1, 4, 2, 1, 4, 1, 1, 4, 4, 4, 3, 2,
       4, 4, 4, 2, 4, 1, 1, 3, 1, 3, 2, 4, 3, 1, 3,

In [11]:
dataset = (
    dataset_from_tensors
    .batch(512*20)
    .prefetch(tf.data.AUTOTUNE)
)


In [12]:
raw_predictions =[]

for i in tqdm(dataset):
    pred = trainer.model.predict(i, verbose=False,batch_size=64)
    for k in ['v','d','j']:
        pred[k+'_segment'] = pred[k+'_segment'].astype(np.float16)
    raw_predictions.append(pred)

  0%|          | 0/270 [00:00<?, ?it/s]

Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: module 'gast' has no attribute 'Index'
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert
Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: module 'gast' has no attribute 'Index'
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


2023-10-24 10:19:43.258444: I tensorflow/compiler/mlir/mlir_graph_optimization_pass.cc:116] None of the MLIR optimization passes are enabled (registered 2)
2023-10-24 10:19:43.259210: I tensorflow/core/platform/profile_utils/cpu_utils.cc:112] CPU Frequency: 2100000000 Hz
2023-10-24 10:19:44.637818: I tensorflow/stream_executor/platform/default/dso_loader.cc:49] Successfully opened dynamic library libcublas.so.10
2023-10-24 10:19:46.932764: I tensorflow/stream_executor/platform/default/dso_loader.cc:49] Successfully opened dynamic library libcudnn.so.7
2023-10-24 10:20:16.721341: E tensorflow/stream_executor/cuda/cuda_dnn.cc:336] Could not create cudnn handle: CUDNN_STATUS_INTERNAL_ERROR
2023-10-24 10:20:16.734418: E tensorflow/stream_executor/cuda/cuda_dnn.cc:336] Could not create cudnn handle: CUDNN_STATUS_INTERNAL_ERROR
2023-10-24 10:20:16.746690: E tensorflow/stream_executor/cuda/cuda_dnn.cc:336] Could not create cudnn handle: CUDNN_STATUS_INTERNAL_ERROR
2023-10-24 10:20:16.757958: 

UnknownError:  Failed to get convolution algorithm. This is probably because cuDNN failed to initialize, so try looking to see if a warning log message was printed above.
	 [[node v_deep_j_allign_experimental_single_beam_conv_segmentation_residual_rf/conv1d_and__batch_norm/conv1d/conv1d (defined at home/bcrlab/eisenbr2/AlignAIRR/VDeepJLayers.py:338) ]] [Op:__inference_predict_function_7069]

Function call stack:
predict_function


In [259]:
v_segment,d_segment,j_segment,v_allele,d_allele,j_allele = [],[],[],[],[],[]

In [260]:
for i in raw_predictions:
    v_segment.append(i['v_segment'])
    d_segment.append(i['d_segment'])
    j_segment.append(i['j_segment'])
    
    v_allele.append(i['v_allele'])
    d_allele.append(i['d_allele'])
    j_allele.append(i['j_allele'])

In [261]:
v_segment = np.vstack(v_segment)
d_segment = np.vstack(d_segment)
j_segment = np.vstack(j_segment)

v_allele = np.vstack(v_allele)
d_allele = np.vstack(d_allele)
j_allele = np.vstack(j_allele)

# Control And Query Different Stage of the Model

In [ ]:

random_d = random.choice(locus[0]['D'])
random_j = random.choice(locus[0]['J'])

generated_seq = generate_sequence_spesific(vo_dict[allele],random_d,random_j,data_dict, mutate=True,
                            mutation_rate=0.01,shm_flat=False, flat_usage='allele')

I = generated_seq.mutated_seq
I = trainer.train_dataset.tokenize_sequences([I])

with tf.GradientTape() as tape:
    # Cast the input image as a TensorFlow tensor, and watch it
    inputs = tf.cast(I, tf.float32)
    tape.watch(inputs)

    # Forward pass through the model
    # STEP 1 : Produce embeddings for the input sequence
    input_seq = trainer.model.reshape_and_cast_input(inputs)
    concatenated_input_embedding = trainer.model.concatenated_input_embedding(input_seq)

    # Residual
    residual_connection_segmentation_conv = trainer.model.residual_connection_segmentation_conv_x_to_1(concatenated_input_embedding)
    residual_connection_segmentation_max_pool = trainer.model.residual_connection_segmentation_max_pool_x_to_1(residual_connection_segmentation_conv)

    # STEP 2: Run Embedded sequence through 1D convolution to distill temporal features
    conv_layer_segmentation_1 = trainer.model.conv_layer_segmentation_1(concatenated_input_embedding)
    conv_layer_segmentation_1_res = trainer.model.residual_connection_segmentation_add_x_to_1([conv_layer_segmentation_1, residual_connection_segmentation_max_pool])
    conv_layer_segmentation_1_res = trainer.model.residual_connection_segmentation_activation_x_to_1(conv_layer_segmentation_1_res)

    conv_layer_segmentation_2 = trainer.model.conv_layer_segmentation_2(conv_layer_segmentation_1_res)

    # residual 2
    conv_layer_segmentation_2_res = trainer.model.residual_connection_segmentation_max_pool_1_to_3(conv_layer_segmentation_1_res)
    conv_layer_segmentation_2_res = trainer.model.residual_connection_segmentation_add_1_to_3([conv_layer_segmentation_2_res,conv_layer_segmentation_2])
    conv_layer_segmentation_2_res = trainer.model.residual_connection_segmentation_activation_1_to_3(conv_layer_segmentation_2_res)


    conv_layer_segmentation_3 = trainer.model.conv_layer_segmentation_3(conv_layer_segmentation_2_res)

    # residual 3
    conv_layer_segmentation_3_res = trainer.model.residual_connection_segmentation_max_pool_2_to_4(
        conv_layer_segmentation_2_res)
    conv_layer_segmentation_3_res = trainer.model.residual_connection_segmentation_add_2_to_4(
        [conv_layer_segmentation_3_res, conv_layer_segmentation_3])
    conv_layer_segmentation_3_res = trainer.model.residual_connection_segmentation_activation_2_to_4(
        conv_layer_segmentation_3_res)

    conv_layer_segmentation_4 = trainer.model.conv_layer_segmentation_4(conv_layer_segmentation_3_res)

    # residual 4
    conv_layer_segmentation_5_res = trainer.model.residual_connection_segmentation_max_pool_3_to_5(
        conv_layer_segmentation_3_res)
    conv_layer_segmentation_5_res = trainer.model.residual_connection_segmentation_add_3_to_5(
        [conv_layer_segmentation_5_res, conv_layer_segmentation_4])
    conv_layer_segmentation_5_res = trainer.model.residual_connection_segmentation_activation_3_to_5(
        conv_layer_segmentation_5_res)

    last_conv_layer = trainer.model.conv_layer_segmentation_5(conv_layer_segmentation_5_res)

    # residual 5
    conv_layer_segmentation_d_res = trainer.model.residual_connection_segmentation_max_pool_5_to_d(
        conv_layer_segmentation_5_res)
    conv_layer_segmentation_d_res = trainer.model.residual_connection_segmentation_add_5_to_d(
        [conv_layer_segmentation_d_res, last_conv_layer])
    conv_layer_segmentation_d_res = trainer.model.residual_connection_segmentation_activation_5_to_d(
        conv_layer_segmentation_d_res)


    # STEP 3 : Flatten The Feature Derived from the 1D conv layers
    concatenated_signals = conv_layer_segmentation_d_res
    concatenated_signals = trainer.model.segmentation_feature_flatten(concatenated_signals)
    concatenated_signals = trainer.model.initial_feature_map_dropout(concatenated_signals)
    # STEP 4 : Predict The Intervals That Contain The V,D and J Genes using (V_start,V_end,D_Start,D_End,J_Start,J_End)
    v_segment, d_segment, j_segment = trainer.model.predict_segments(concatenated_signals)

    reshape_masked_sequence_v = trainer.model.v_mask_reshape(v_segment)
    reshape_masked_sequence_d = trainer.model.d_mask_reshape(d_segment)
    reshape_masked_sequence_j = trainer.model.j_mask_reshape(j_segment)

    masked_sequence_v = trainer.model.v_mask_gate([reshape_masked_sequence_v,concatenated_input_embedding])
    masked_sequence_d = trainer.model.d_mask_gate([reshape_masked_sequence_d,concatenated_input_embedding])
    masked_sequence_j = trainer.model.j_mask_gate([reshape_masked_sequence_j,concatenated_input_embedding])


    # Pass The Embeddings Generated Above Thorough 2D Convolutional Feature Extractor Layer
    v_feature_map = trainer.model._encode_masked_v_signal(masked_sequence_v)
    d_feature_map = trainer.model._encode_masked_d_signal(masked_sequence_d)
    j_feature_map = trainer.model._encode_masked_j_signal(masked_sequence_j)

    # STEP 8: Predict The V,D and J genes
    v_allele, d_allele, j_allele = trainer.model._predict_vdj_set(v_feature_map, d_feature_map, j_feature_map)

    # Get the prediction of the target category
    target_category_loss = v_allele[0][v_allele_call_ohe[allele]]
    # Compute the gradient of the target class prediction with respect to the feature maps
    gradients = tape.gradient(target_category_loss, reshape_masked_sequence_v)

